# Taxi Demand Forecasting

Forecasting the number of airport taxi orders one hour ahead.

**Result:** CatBoost achieved test RMSE = 39.54, compared with 58.8 for the naive baseline.

**Methods:** time series, lag features, rolling statistics, TimeSeriesSplit, CatBoost, RMSE.

> This portfolio version removes course-review correspondence and repetitive instructional text. The analysis, models, and reported metrics are based on the original completed project. The source datasets are not included in this repository.


## 1. Setup and data preparation

Orders are indexed by timestamp, sorted chronologically, and resampled to hourly frequency.


In [2]:
from statsmodels.tsa.seasonal import seasonal_decompose
from sklearn.model_selection import (GridSearchCV,
                                     train_test_split,
                                     cross_val_score,
                                     TimeSeriesSplit
                                     )
from lightgbm import LGBMRegressor
from catboost import CatBoostRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.metrics import mean_squared_error
import pandas as pd

import seaborn as sns

import sweetviz as sv

import matplotlib.pyplot as plt

import numpy as np

import warnings
warnings.filterwarnings('ignore')


In [3]:
try:
    data = pd.read_csv(
        '/Users/kolotukhin.md/Downloads/jupyter_notebook/10/taxi.csv', index_col=[0], parse_dates=[0])
except:
    data = pd.read_csv(
        'https://code.s3.yandex.net/datasets/taxi.csv', index_col=[0], parse_dates=[0])

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)


In [4]:
print(data.index.is_monotonic)


True


In [5]:
# data.sort_index(inplace=True)


In [6]:
#data = data.resample('1H').mean()


In [7]:
data = data.resample('1H').sum()


In [8]:
data.info()


<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 4416 entries, 2018-03-01 00:00:00 to 2018-08-31 23:00:00
Freq: H
Data columns (total 1 columns):
 #   Column      Non-Null Count  Dtype
---  ------      --------------  -----
 0   num_orders  4416 non-null   int64
dtypes: int64(1)
memory usage: 69.0 KB


In [9]:
print(data.head())


                     num_orders
datetime                       
2018-03-01 00:00:00         124
2018-03-01 01:00:00          85
2018-03-01 02:00:00          71
2018-03-01 03:00:00          66
2018-03-01 04:00:00          43


In [10]:

report = sv.analyze([data, "General_information_1.0"])
#report.show_html('general_information_1.0.html')
report.show_notebook('general_information_1.0.html')


## 2. Time-series analysis

Trend, seasonality, and rolling statistics are inspected to guide feature construction and validation.


In [11]:
data_graph_H = data.copy()


In [12]:

decomposed = seasonal_decompose(data_graph_H)

plt.figure(figsize=(15, 15))
plt.subplot(311)


decomposed.trend.plot(ax=plt.gca(), fontsize=14)
plt.title('Trend', fontsize=20)
plt.xlabel('Date', fontsize=18)
plt.ylabel('Number of orders', fontsize=18)
plt.grid()
plt.subplot(312)
decomposed.seasonal.plot(ax=plt.gca(), fontsize=14)
plt.title('Seasonality', fontsize=20)
plt.xlabel('Date', fontsize=18)
plt.grid()
plt.subplot(313)
decomposed.resid.plot(ax=plt.gca(), fontsize=14)
plt.title('Residuals', fontsize=20)
plt.xlabel('Date', fontsize=18)
plt.grid()
plt.tight_layout()


<Figure size 1080x1080 with 3 Axes>

In [13]:
data_graph_D = data_graph_H.resample('1D').sum()


In [14]:

decomposed = seasonal_decompose(data_graph_D)

plt.figure(figsize=(15, 15))
plt.subplot(311)


decomposed.trend.plot(ax=plt.gca(), fontsize=14)
plt.title('Trend', fontsize=20)
plt.xlabel('Date', fontsize=18)
plt.ylabel('Number of orders', fontsize=18)
plt.grid()
plt.subplot(312)
decomposed.seasonal.plot(ax=plt.gca(), fontsize=14)
plt.title('Seasonality', fontsize=20)
plt.xlabel('Date', fontsize=18)
plt.grid()
plt.subplot(313)
decomposed.resid.plot(ax=plt.gca(), fontsize=14)
plt.title('Residuals', fontsize=20)
plt.xlabel('Date', fontsize=18)
plt.grid()
plt.tight_layout()


<Figure size 1080x1080 with 3 Axes>

In [15]:
data_graph_H = data_graph_H['2018-07-01':'2018-07-07']


decomposed = seasonal_decompose(data_graph_H)

plt.figure(figsize=(15, 15))
plt.subplot(311)


decomposed.trend.plot(ax=plt.gca(), fontsize=14)
plt.title('Trend', fontsize=20)
plt.xlabel('Date', fontsize=18)
plt.ylabel('Number of orders', fontsize=18)
plt.grid()
plt.subplot(312)
decomposed.seasonal.plot(ax=plt.gca(), fontsize=14)
plt.title('Seasonality', fontsize=20)
plt.xlabel('Date', fontsize=18)
plt.grid()
plt.subplot(313)
decomposed.resid.plot(ax=plt.gca(), fontsize=14)
plt.title('Residuals', fontsize=20)
plt.xlabel('Date', fontsize=18)
plt.grid()
plt.tight_layout()


<Figure size 1080x1080 with 3 Axes>

In [4]:
#data_graph_H = data_graph_H['2018-07-01':'2018-07-31'].resample('1D').sum()
data_graph_day = data['2018-07-01':'2018-07-02']


decomposed = seasonal_decompose(data_graph_day)

plt.figure(figsize=(15, 15))
plt.subplot(311)


decomposed.trend.plot(ax=plt.gca(), fontsize=14)
plt.title('Trend', fontsize=20)
plt.xlabel('Date', fontsize=18)
plt.ylabel('Number of orders', fontsize=18)
plt.grid()
plt.subplot(312)
decomposed.seasonal.plot(ax=plt.gca(), fontsize=14)
plt.title('Seasonality', fontsize=20)
plt.xlabel('Date', fontsize=18)
plt.grid()
plt.subplot(313)
decomposed.resid.plot(ax=plt.gca(), fontsize=14)
plt.title('Residuals', fontsize=20)
plt.xlabel('Date', fontsize=18)
plt.grid()
plt.tight_layout()


In [17]:
print()


In [18]:
data_stat = data.copy()


In [19]:
data_stat['mean'] = data_stat['num_orders'].rolling(20).mean()
data_stat['std'] = data_stat['num_orders'].rolling(20).std()


In [20]:
#data_stat.plot()


In [21]:
data_stat.plot(fontsize=14, figsize=(15,10))
plt.title('Historical airport taxi demand', fontsize=20)
plt.xlabel('Date', fontsize=18)
plt.ylabel('Number of orders', fontsize=18)
plt.legend(data_stat, prop = {'size':20}, loc='upper left')
plt.grid()
plt.show()


<Figure size 1080x720 with 1 Axes>

In [22]:
from statsmodels.tsa.stattools import adfuller

ts = data['num_orders']
st_test = adfuller(ts, regression='ctt')

print('If the p-value is below 0.05, the series is treated as stationary')
if st_test[1] < 0.05:
    print('The series is stationary')
else:
    print('The series is non-stationary')


In [23]:

#data_stat = data_stat - data_stat.shift()

#data_stat.plot(fontsize=14, figsize=(15,10))
#plt.legend(data_stat, prop = {'size':20}, loc='upper left')
#plt.grid()
#plt.show()


In [24]:
#data = data - data.shift()


In [25]:
#data['rolling_mean'] = data.rolling(10).mean()
# data.plot()


In [26]:
data.info()


<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 4416 entries, 2018-03-01 00:00:00 to 2018-08-31 23:00:00
Freq: H
Data columns (total 1 columns):
 #   Column      Non-Null Count  Dtype
---  ------      --------------  -----
 0   num_orders  4416 non-null   int64
dtypes: int64(1)
memory usage: 198.0 KB


In [27]:
#print(dfg)


## 3. Feature engineering and model selection

Calendar fields, lags, and rolling means are created without using future observations. TimeSeriesSplit preserves temporal order during cross-validation.


In [28]:
def make_features(data, max_lag, rolling_mean_size):
    data['hour'] = data.index.hour
    data['day'] = data.index.day
    data['day_of_week'] = data.index.dayofweek
    data['days_in_month'] = data.index.days_in_month

    for lag in range(1, max_lag + 1):
        data['lag_{}'.format(lag)] = data['num_orders'].shift(lag)

        data['rolling_mean'] = data['num_orders'].shift().rolling(
            rolling_mean_size).mean()


make_features(data, 60, 30)

train, test = train_test_split(data, shuffle=False, test_size=0.1)

train = train.dropna()

features_train = train.drop('num_orders', axis=1)
target_train = train['num_orders']

features_test = test.drop('num_orders', axis=1)
target_test = test['num_orders']


In [29]:
#features_train.info()


In [30]:
print()


In [31]:
tscv = TimeSeriesSplit(n_splits=10)

regressor = LinearRegression()
print('# Train for root_mean_squared_error')
print()
cv_RMSE_LR = (cross_val_score(regressor,
                              features_train,
                              target_train,
                              cv=tscv,
                              scoring='neg_mean_squared_error').mean() * -1) ** 0.5
print('Mean RMSE from CV of LinearRegression:', round(cv_RMSE_LR, 2))


# Train for root_mean_squared_error

Mean RMSE from CV of LinearRegression: 160603051045.48


In [32]:
tscv = TimeSeriesSplit(n_splits=10)

regressor = Ridge()
hyperparams = [{'solver': ['auto', 'svd', 'cholesky', 'lsqr', 'sparse_cg']}]


print('# Tuning hyper-parameters for root_mean_squared_error')
print()
clf = GridSearchCV(regressor,
                   hyperparams,
                   scoring='neg_root_mean_squared_error',
                   cv=tscv
                   )
clf.fit(features_train, target_train)
print("Best parameters set found on development set:")
print()
print(clf.best_params_)
print()
print(clf.cv_results_)
print()
cv_RMSE_R = clf.best_score_ * -1
print('best_score RMSE:', round(cv_RMSE_R, 2))


# Tuning hyper-parameters for root_mean_squared_error

Best parameters set found on development set:

{'solver': 'lsqr'}

{'mean_fit_time': array([0.00603616, 0.00834033, 0.00478418, 0.00558562, 0.00633128]), 'std_fit_time': array([0.00296315, 0.00219525, 0.00123483, 0.00097679, 0.00079711]), 'mean_score_time': array([0.00196979, 0.00190492, 0.00196872, 0.00203981, 0.00204871]), 'std_score_time': array([0.00023323, 0.0001825 , 0.00030144, 0.00021323, 0.00024384]), 'param_solver': masked_array(data=['auto', 'svd', 'cholesky', 'lsqr', 'sparse_cg'],
             mask=[False, False, False, False, False],
       fill_value='?',
            dtype=object), 'params': [{'solver': 'auto'}, {'solver': 'svd'}, {'solver': 'cholesky'}, {'solver': 'lsqr'}, {'solver': 'sparse_cg'}], 'split0_test_score': array([-21.39540478, -21.39540478, -21.39540478, -21.38538336,
       -21.39726198]), 'split1_test_score': array([-19.52128924, -19.52128924, -19.52128924, -19.52684589,
       -19.49331911]), 'split2_

In [33]:
tscv = TimeSeriesSplit(n_splits=10)

regressor = DecisionTreeRegressor()
hyperparams = [{'criterion': ['squared_error'],
                'max_depth': np.arange(1, 20, 1),
                'max_features': np.arange(1, 13, 1),
                'min_samples_leaf': [x for x in range(1, 20, 1)],
                'random_state': [12345]}]

print('# Tuning hyper-parameters for root_mean_squared_error')
print()
clf = GridSearchCV(regressor,
                   hyperparams,
                   scoring='neg_root_mean_squared_error',
                   n_jobs=-1,
                   cv=tscv
                  )
clf.fit(features_train, target_train)
print("Best parameters set found on development set:")
print()
print(clf.best_params_)
print()
cv_RMSE_DTR = clf.best_score_ * -1
print('best_score RMSE:', round(cv_RMSE_DTR, 2))


# Tuning hyper-parameters for root_mean_squared_error

Best parameters set found on development set:

{'criterion': 'squared_error', 'max_depth': 14, 'max_features': 11, 'min_samples_leaf': 19, 'random_state': 12345}

best_score RMSE: 27.22


In [34]:
tscv = TimeSeriesSplit(n_splits=10)

regressor = RandomForestRegressor()
hyperparams = [{'criterion': ['squared_error'],
                'max_depth': [x for x in range(2, 15, 3)],
                'n_estimators': np.arange(1, 201, 25),
                'random_state': [12345]}]


print('# Tuning hyper-parameters for root_mean_squared_error')
print()
clf = GridSearchCV(regressor,
                   hyperparams,
                   scoring='neg_root_mean_squared_error',
                   n_jobs=-1,
                   cv=tscv
                  )
clf.fit(features_train, target_train)
print("Best parameters set found on development set:")
print()
print(clf.best_params_)
print()
cv_RMSE_DTR = clf.best_score_ * -1
print('best_score RMSE:', round(cv_RMSE_DTR, 2))


# Tuning hyper-parameters for root_mean_squared_error

Best parameters set found on development set:

{'criterion': 'squared_error', 'max_depth': 14, 'n_estimators': 176, 'random_state': 12345}

best_score RMSE: 23.47


In [35]:
tscv = TimeSeriesSplit(n_splits=10)

regressor = CatBoostRegressor()
hyperparams = [{'learning_rate': np.arange(0.1, 1.1, 0.1),
                'random_state': [12345],
                'verbose': [False]}]

print('# Tuning hyper-parameters for root_mean_squared_error')
print()
clf = GridSearchCV(regressor,
                   hyperparams,
                   scoring='neg_root_mean_squared_error',
                   cv=tscv
                  )
clf.fit(features_train, target_train)
print("Best parameters set found on development set:")
print()
print(clf.best_params_)
print()
cv_RMSE_CBR = clf.best_score_ * -1
print('best_score RMSE:', round(cv_RMSE_CBR, 2))


# Tuning hyper-parameters for root_mean_squared_error

Best parameters set found on development set:

{'learning_rate': 0.1, 'random_state': 12345, 'verbose': False}

best_score RMSE: 22.84


In [36]:
tscv = TimeSeriesSplit(n_splits=10)

regressor = LGBMRegressor()
hyperparams = [{'num_leaves': np.arange(2, 10, 1),
                'learning_rate': np.arange(0.1, 1.1, 0.1),
                'random_state': [12345]}]

print('# Tuning hyper-parameters for root_mean_squared_error')
print()
clf = GridSearchCV(regressor,
                   hyperparams,
                   scoring='neg_root_mean_squared_error',
                   cv=tscv
                  )
clf.fit(features_train, target_train)
print("Best parameters set found on development set:")
print()
print(clf.best_params_)
print()
cv_RMSE_LGBMR = clf.best_score_ * -1
print('best_score RMSE:', round(cv_RMSE_LGBMR, 2))


# Tuning hyper-parameters for root_mean_squared_error

Best parameters set found on development set:

{'learning_rate': 0.1, 'num_leaves': 8, 'random_state': 12345}

best_score RMSE: 23.21


## 4. Held-out evaluation

The selected CatBoost model is compared with a one-step naive forecast on the final chronological test interval.


In [42]:
model = CatBoostRegressor(learning_rate=0.1,
                          random_state=12345,
                          verbose=False
                          )

model.fit(features_train, target_train)

predict = model.predict(features_test)

RMSE = round((mean_squared_error(target_test, predict) ** 0.5), 2)
print('CatBoostRegressor RMSE :', RMSE)


CatBoostRegressor RMSE : 39.54


In [47]:
pred_previous = target_test.shift()
pred_previous.iloc[0] = target_train.iloc[-1]
rmse_previous = mean_squared_error(target_test, pred_previous)**0.5
print("Naive previous-value RMSE:", round(rmse_previous, 2))


## Conclusion

The CatBoost model reduced RMSE from 58.8 for the naive forecast to 39.54 on the held-out period. The result supports using lagged demand and calendar structure for short-horizon staffing forecasts, while performance should be monitored for seasonal drift.
